In [ ]:
import pandas as pd
import numpy as np
import re, json
from datetime import datetime

RAW_DIR = "sepadu_data"          # folder with the two .xlsx files
OUT_DIR = "data_export"          # where cleaned outputs will be written
import os
os.makedirs(OUT_DIR, exist_ok=True)

SMK_PATH = f"{RAW_DIR}/sepadu_maklumat_guru-SMK.xlsx"
SK_PATH  = f"{RAW_DIR}/sepadu_maklumat_guru-SK.xlsx"

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [ ]:
df_smk_raw = pd.read_excel(SMK_PATH)
df_sk_raw  = pd.read_excel(SK_PATH)

print("SMK:", df_smk_raw.shape)
print("SK :", df_sk_raw.shape)

In [ ]:
def basic_clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.dropna(axis=1, how="all").copy()
    str_cols = df.select_dtypes(include="object").columns
    for c in str_cols:
        df[c] = (
            df[c].astype(str).str.strip()
            .replace({"nan": np.nan, "None": np.nan, "": np.nan, "TIADA MAKLUMAT": np.nan})
        )
    return df

df_smk = basic_clean(df_smk_raw)
df_sk  = basic_clean(df_sk_raw)

print("SMK after clean:", df_smk.shape)
print("SK  after clean:", df_sk.shape)

In [ ]:
df = pd.concat([df_smk, df_sk], ignore_index=True)
print("Combined:", df.shape)

df["NEGERI_ALAMAT"] = df["NEGERI.1"].fillna(df["NEGERI"])
df["POSKOD_ALAMAT"] = df["POSKOD"].fillna(df.get("POSKOD.1"))
df["BANDAR_ALAMAT"] = df["BANDAR"].fillna(df.get("BANDAR.1"))
df["TELEFON_UTAMA"] = df["NO. TELEFON"].fillna(df.get("NO. TELEFON.1"))

df["JENIS SEKOLAH"].value_counts(dropna=False)

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate NO. KP:", df['NO. KP'].duplicated().sum())
print("Missing NAMA:", df['NAMA'].isna().sum())
print("Missing TARIKH LAHIR:", df['TARIKH LAHIR'].isna().sum())
print("Missing GAJI:", df['GAJI'].isna().sum())

null_pct = (df.isna().mean() * 100).sort_values(ascending=False)
null_pct[null_pct > 0].head(20)

In [ ]:
def parse_ddmmyyyy(s):
    """'08/04/1981' -> '1981-04-08'"""
    if pd.isna(s):
        return None
    try:
        return datetime.strptime(str(s).strip(), "%d/%m/%Y").strftime("%Y-%m-%d")
    except Exception:
        return None

def parse_salary(s):
    """'RM8976.06' -> 8976.06"""
    if pd.isna(s):
        return None
    m = re.search(r"[\d.]+", str(s).replace(",", ""))
    return float(m.group()) if m else None

def format_ic(s):
    """'900101015555' -> '900101-01-5555' (contoh sahaja)"""
    if pd.isna(s):
        return ""
    digits = re.sub(r"\D", "", str(s))
    if len(digits) != 12:
        return str(s)
    return f"{digits[0:6]}-{digits[6:8]}-{digits[8:12]}"

def clean_phone(s):
    if pd.isna(s):
        return ""
    digits = re.sub(r"\D", "", str(s))
    if len(digits) >= 10:
        return f"{digits[:3]}-{digits[3:7]} {digits[7:]}"
    return str(s)

def yn_bool(s):
    if pd.isna(s):
        return False
    return str(s).strip().upper() == "YA"

In [ ]:
def safe_str(s, title=False):
    """Returns None for missing values, otherwise a clean stripped string."""
    if pd.isna(s):
        return None
    s = str(s).strip()
    return s.title() if title else s

In [ ]:
TODAY_YEAR = datetime.now().year

def build_teacher(row, idx):
    dob = parse_ddmmyyyy(row.get("TARIKH LAHIR"))
    age = TODAY_YEAR - int(dob[:4]) if dob else None

    subjects = []
    for i in (1, 2, 3):
        name = row.get(f"SUBJEK AJAR{i}")
        hrs = row.get(f"BIL WAKTU{i}")
        if pd.notna(name):
            subjects.append({"name": str(name).title(), "hours": int(hrs) if pd.notna(hrs) else 0})

    addr_parts = [row.get("ALAMAT SEMASA 1"), row.get("ALAMAT SEMASA 2"), row.get("ALAMAT SEMASA 3")]
    addr_line = ", ".join(str(a) for a in addr_parts if pd.notna(a))
    poskod = row.get("POSKOD_ALAMAT")
    poskod_str = str(int(poskod)) if pd.notna(poskod) else ""
    full_address = f"{addr_line}, {poskod_str} {row.get('BANDAR_ALAMAT','')}, {row.get('NEGERI_ALAMAT','')}".strip(", ")

    kedudukan = str(row.get("KEDUDUKAN PERKHIDMATAN", ""))
    service_status = "Aktif" if kedudukan.upper() == "BERKHIDMAT DI SEKOLAH" else "Tidak Aktif"

    taraf = str(row.get("TARAF JAWATAN", "")).upper()
    if "KONTRAK" in taraf:
        employment_status = "Kontrak"
    elif "COS" in taraf or "SANDARAN" in taraf:
        employment_status = "Sandaran"
    else:
        employment_status = "Tetap"

    induksi_raw = str(row.get("STATUS INDUKSI", ""))
    status_induksi = "Selesai" if induksi_raw in ("Telah Hadir Dan Lulus", "Dikecualikan") else "Dalam Tempoh Induksi"

    sah = str(row.get("SAH PERKHIDMATAN", "")).upper()
    sah_perkhidmatan = "Disahkan" if sah == "YA" else "Belum Disahkan"

    ikhtisas_raw = row.get("IKHTISAS")
    ikhtisas = bool(pd.notna(ikhtisas_raw) and str(ikhtisas_raw).strip().upper() != "TIADA")

    tugas = None
    for i in (1, 2, 3, 4, 5):
        v = row.get(f"TUGAS KHAS {i}")
        if pd.notna(v):
            tugas = str(v).title()
            break
    tugas = tugas or "Tiada"

    vaccinated = yn_bool(row.get("STATUS VAKSIN"))

    return {
        "id": idx,
        "name": str(row.get("NAMA", "")).title(),
        "gender": safe_str(row.get("JANTINA")),
        "icNumber": format_ic(row.get("NO. KP")),
        "school": safe_str(row.get("NAMA SEKOLAH"), title=True),
        "schoolType": safe_str(row.get("JENIS SEKOLAH")),
        "position": str(row.get("JAWATAN", "")).title(),
        "gredHakiki": safe_str(row.get("GRED HAKIKI")),
        "gredPenyandang": safe_str(row.get("GRED PENYANDANG")),
        "employmentStatus": employment_status,
        "serviceStatus": service_status,
        "dateOfBirth": dob,
        "maritalStatus": safe_str(row.get("STATUSKAHWIN"), title=True),
        "race": safe_str(row.get("KAUM"), title=True),
        "religion": safe_str(row.get("AGAMA"), title=True),
        "nationality": safe_str(row.get("WARGANEGARA")),
        "firstAppointmentDate": parse_ddmmyyyy(row.get("TARIKH LANTIKAN PERTAMA")),
        "qualification": safe_str(row.get("IKHTISAS")) or safe_str(row.get("AKADEMIK TERTINGGI")),
        "opsyen1": safe_str(row.get("OPSYEN 1")),
        "opsyen2": safe_str(row.get("OPSYEN 2")),
        "subjects": subjects,
        "phone": clean_phone(row.get("TELEFON_UTAMA")),
        "email": row.get("EMEL"),
        "address": full_address,
        "vaccinated": vaccinated,
        "vaccineType": safe_str(row.get("JENIS VAKSIN")) if vaccinated else None,
        "vaccineDose1": parse_ddmmyyyy(row.get("TARIKH DOS1")) if vaccinated else None,
        "vaccineDose2": parse_ddmmyyyy(row.get("TARIKH DOS2")) if vaccinated else None,
        "salary": parse_salary(row.get("GAJI")),
        "age": age,
        "district": safe_str(row.get("DAERAH SEMASA")),
        "retirementDate": parse_ddmmyyyy(row.get("TARIKH BERSARA")),
        "ikhtisas": ikhtisas,
        "statusInduksi": status_induksi,
        "sahPerkhidmatan": sah_perkhidmatan,
        "guruCemerlang": yn_bool(row.get("GURU CEMERLANG")),
        "guruCemerlangBidang": row.get("BIDANG GURU CEMERLANG") if pd.notna(row.get("BIDANG GURU CEMERLANG")) else "",
        "tugasKhas": tugas,
        "kodJawatanDiisi": str(row.get("PENGISIAN", "")).upper() not in ("", "NAN"),
    }

teachers = [build_teacher(r, i + 1) for i, r in df.iterrows()]
print(f"Built {len(teachers)} teacher records")
teachers[0]

In [ ]:
def sorted_unique(series):
    return sorted(x for x in series.dropna().unique().tolist())

schools_by_type = {}
for jenis, grp in df.groupby("JENIS SEKOLAH"):
    schools_by_type[jenis] = sorted_unique(grp["NAMA SEKOLAH"].str.title())

lookups = {
    "schoolTypes": sorted_unique(df["JENIS SEKOLAH"]),
    "schoolsByType": schools_by_type,
    "subjects": sorted_unique(pd.concat([df["SUBJEK AJAR1"], df["SUBJEK AJAR2"], df["SUBJEK AJAR3"]]).str.title()),
    "grades": sorted_unique(pd.concat([df["GRED HAKIKI"], df["GRED PENYANDANG"]])),
    "races": sorted_unique(df["KAUM"].str.title()),
    "religions": sorted_unique(df["AGAMA"].str.title()),
    "maritalStatuses": sorted_unique(df["STATUSKAHWIN"].str.title()),
    "districts": sorted_unique(df["DAERAH SEMASA"]),
    "employmentStatuses": ["Tetap", "Kontrak", "Sandaran"],
}
{k: (v if not isinstance(v, dict) else f"{len(v)} school types") for k, v in lookups.items()}

In [ ]:
with open(f"{OUT_DIR}/teachers.json", "w", encoding="utf-8") as f:
    json.dump(teachers, f, ensure_ascii=False, indent=2)

with open(f"{OUT_DIR}/lookups.json", "w", encoding="utf-8") as f:
    json.dump(lookups, f, ensure_ascii=False, indent=2)

print("Wrote:", f"{OUT_DIR}/teachers.json", f"({len(teachers)} records)")
print("Wrote:", f"{OUT_DIR}/lookups.json")